# 🏥 Hospital DB Migration — MySQL to Excel ETL Pipeline

---

## Overview

This notebook connects directly to the `hospital_db` MySQL database, runs six analytical queries across multiple joined tables, and exports the results as a  formatted multi-sheet Excel workbook.



---
## Section 1 — Install Dependencies

Install the required libraries if not already present.  
- `mysql-connector-python` — official MySQL connector (installed first as a fallback)
- `pymysql` — pure Python MySQL client used in this project (more reliable with stored procedures in Jupyter)

In [ ]:
pip install mysql-connector-python --upgrade

Defaulting to user installation because normal site-packages is not writeable
  Using cached mysql_connector_python-9.6.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
Using cached mysql_connector_python-9.6.0-cp311-cp311-win_amd64.whl (16.5 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install pymysql

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Section 2 — Import Libraries

In [ ]:
import pymysql
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import os

print("All imports successful ✅")

All imports successful ✅


---
## Section 3 — Connect to MySQL

Establishes a live connection to the local `hospital_db` MySQL database using PyMySQL.

In [ ]:
connection = pymysql.connect(
    host     = "localhost",
    port     = 3306,
    user     = "root",
    password = "Sonu@321",
    database = "hospital_db"
)

print("Connected to MySQL ✅")

Connected to MySQL ✅


---
## Section 4 — Helper Functions

Two reusable functions that keep the query section clean:

- **`run_procedure()`** — calls a stored procedure by name and returns results as a DataFrame. Handles PyMySQL's `nextset()` behavior, which differs from `mysql-connector`.
- **`run_query()`** — runs a raw SQL string and returns results as a DataFrame. Used for all analytical queries in this notebook.

In [ ]:
def run_procedure(connection, procedure_name, params=()):
    cursor = connection.cursor()
    cursor.callproc(procedure_name, params)

    columns = []
    results = []

    # PyMySQL stores procedure results differently than mysql-connector
    # We need to fetch from nextset()
    cursor.execute(f"CALL {procedure_name}({','.join(['%s']*len(params))})", params)
    columns = [col[0] for col in cursor.description]
    results = cursor.fetchall()
    cursor.close()

    df = pd.DataFrame(results, columns=columns)
    print(f"✅ {procedure_name} → {len(df)} rows returned")
    return df


def run_query(connection, query, label):
    df = pd.read_sql(query, connection)
    print(f"✅ {label} → {len(df)} rows returned")
    return df

print("Helper functions ready ✅")

Helper functions ready ✅



---
## Section 5 — Run Analytical Queries

Executes six multi-table JOIN queries against `hospital_db` and stores each result as a pandas DataFrame

In [ ]:
print("Running all procedures...\n")

df_revenue_all = run_query(
    connection,
    """
    SELECT
        YEAR(B.billDate)                    AS year,
        MONTH(B.billDate)                   AS month_num,
        DATE_FORMAT(B.billDate, '%b %Y')    AS month_name,
        D1.name                             AS department,
        COUNT(B.billID)                     AS total_bills,
        ROUND(SUM(B.amount), 2)             AS total_revenue,
        ROUND(AVG(B.amount), 2)             AS avg_bill_amount
    FROM       bills        AS B
    INNER JOIN appointments AS A  ON A.appointmentID = B.appointmentID
    INNER JOIN doctors      AS D  ON D.doctorID      = A.doctorID
    INNER JOIN departments  AS D1 ON D1.departmentID = D.departmentID
    GROUP BY
        YEAR(B.billDate),
        MONTH(B.billDate),
        DATE_FORMAT(B.billDate, '%b %Y'),
        D1.name
    ORDER BY year, month_num, total_revenue DESC
    """,
    "All Monthly Revenue"
)

# Also get monthly totals (for trend line chart)
df_monthly_trend = run_query(
    connection,
    """
    SELECT
        DATE_FORMAT(B.billDate, '%b %Y')   AS month_name,
        YEAR(B.billDate)                   AS year,
        MONTH(B.billDate)                  AS month_num,
        COUNT(B.billID)                    AS total_bills,
        ROUND(SUM(B.amount), 2)            AS total_revenue,
        SUM(CASE WHEN B.paid=1 THEN B.amount ELSE 0 END)  AS collected,
        SUM(CASE WHEN B.paid=0 THEN B.amount ELSE 0 END)  AS uncollected
    FROM bills AS B
    GROUP BY
        DATE_FORMAT(B.billDate, '%b %Y'),
        YEAR(B.billDate),
        MONTH(B.billDate)
    ORDER BY year, month_num
    """,
    "Monthly Trend"
)


# PROCEDURE 2: Doctor Performance 2025
df_doctor_perf = run_query(
    connection,
    """
    SELECT
        D.name                                                   AS doctor_name,
        D.role,
        D1.name                                                  AS department,
        COUNT(A.appointmentID)                                   AS total_appointments,
        SUM(B.amount)                                            AS total_revenue,
        ROUND(AVG(B.amount), 2)                                  AS avg_bill_per_appointment,
        SUM(CASE WHEN B.paid = 1 THEN 1 ELSE 0 END)             AS paid_count,
        SUM(CASE WHEN B.paid = 0 THEN 1 ELSE 0 END)             AS unpaid_count,
        SUM(CASE WHEN A.status = 'Cancelled' THEN 1 ELSE 0 END) AS cancellations
    FROM       doctors      AS D
    JOIN       departments  AS D1 ON D1.departmentID = D.departmentID
    JOIN       appointments AS A  ON A.doctorID      = D.doctorID
    JOIN       bills        AS B  ON B.appointmentID = A.appointmentID
    WHERE YEAR(A.appointmentTime) = 2025
    GROUP BY D.doctorID, D.name, D.role, D1.name
    ORDER BY total_revenue DESC
    """,
    "Doctor Performance 2025"
)

# PROCEDURE 3: Unpaid Bills
df_unpaid = run_query(
    connection,
    """
    SELECT
        D1.name                                              AS department,
        COUNT(B.billID)                                      AS unpaid_bill_count,
        SUM(B.amount)                                        AS unpaid_amount,
        ROUND(
            SUM(B.amount) / (SELECT SUM(amount) FROM bills) * 100
        , 1)                                                 AS pct_of_total_revenue
    FROM       bills        AS B
    JOIN       appointments AS A  ON A.appointmentID = B.appointmentID
    JOIN       doctors      AS D  ON D.doctorID      = A.doctorID
    JOIN       departments  AS D1 ON D1.departmentID = D.departmentID
    WHERE B.paid = 0
    GROUP BY D1.name
    ORDER BY unpaid_amount DESC
    """,
    "Unpaid Bills"
)

# BONUS: Appointment Status Summary
df_appt_summary = run_query(
    connection,
    """
    SELECT
        status,
        COUNT(*)                                        AS total,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS percentage
    FROM appointments
    GROUP BY status
    ORDER BY total DESC
    """,
    "Appointment Summary"
)

# BONUS: Patient Age Groups
df_age_groups = run_query(
    connection,
    """
    SELECT
        CASE
            WHEN TIMESTAMPDIFF(YEAR, dateOfBirth, CURDATE()) BETWEEN 0  AND 18 THEN '0-18'
            WHEN TIMESTAMPDIFF(YEAR, dateOfBirth, CURDATE()) BETWEEN 19 AND 30 THEN '19-30'
            WHEN TIMESTAMPDIFF(YEAR, dateOfBirth, CURDATE()) BETWEEN 31 AND 45 THEN '31-45'
            WHEN TIMESTAMPDIFF(YEAR, dateOfBirth, CURDATE()) BETWEEN 46 AND 60 THEN '46-60'
            WHEN TIMESTAMPDIFF(YEAR, dateOfBirth, CURDATE()) BETWEEN 61 AND 75 THEN '61-75'
            ELSE '75+'
        END                AS age_group,
        COUNT(*)           AS patient_count
    FROM patients
    GROUP BY age_group
    ORDER BY age_group
    """,
    "Patient Age Groups"
)

print("\nAll data collected ✅")

Running all procedures...



C:\Users\Sonu\AppData\Local\Temp\ipykernel_29800\3655380144.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


✅ All Monthly Revenue → 130 rows returned
✅ Monthly Trend → 13 rows returned
✅ Doctor Performance 2025 → 184 rows returned
✅ Unpaid Bills → 10 rows returned
✅ Appointment Summary → 3 rows returned
✅ Patient Age Groups → 6 rows returned

All data collected ✅


In [ ]:
print("=== MONTHLY TREND (all 13 months) ===")
print(df_monthly_trend)

print("\n=== REVENUE ALL MONTHS (top 10 rows) ===")
print(df_revenue_all.head(10))

print("\n=== DOCTOR PERFORMANCE (top 5) ===")
print(df_doctor_perf.head(5))

print("\n=== UNPAID BILLS ===")
print(df_unpaid)

print("\n=== APPOINTMENT SUMMARY ===")
print(df_appt_summary)

print("\n=== PATIENT AGE GROUPS ===")
print(df_age_groups)

=== MONTHLY TREND (all 13 months) ===
   month_name  year  month_num  total_bills  total_revenue  collected  \
0    Jun 2024  2024          6           79      194575.46  113559.63   
1    Jul 2024  2024          7          288      737834.63  524490.04   
2    Aug 2024  2024          8          288      760869.10  507995.87   
3    Sep 2024  2024          9          299      735094.84  541408.00   
4    Oct 2024  2024         10          293      740648.29  476967.52   
5    Nov 2024  2024         11          276      689545.56  476807.88   
6    Dec 2024  2024         12          281      689409.98  480159.72   
7    Jan 2025  2025          1          305      793454.70  524477.48   
8    Feb 2025  2025          2          283      696900.82  485711.84   
9    Mar 2025  2025          3          325      850965.43  581646.51   
10   Apr 2025  2025          4          267      717541.06  506549.14   
11   May 2025  2025          5          302      747518.22  530847.06   
12   Jun 2025

---
## Section 7 — Export to Excel

In [ ]:
# Save
output_path = os.path.join(os.path.expanduser("~"),
                           "Documents",
                           "hospital_report.xlsx")

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_monthly_trend.to_excel(writer,  sheet_name='Monthly Trend',      index=False)
    df_revenue_all.to_excel(writer,    sheet_name='Revenue All Months',  index=False)
    df_doctor_perf.to_excel(writer,    sheet_name='Doctor Performance',  index=False)
    df_unpaid.to_excel(writer,         sheet_name='Unpaid Bills',        index=False)
    df_appt_summary.to_excel(writer,   sheet_name='Appointment Summary', index=False)
    df_age_groups.to_excel(writer,     sheet_name='Patient Age Groups',  index=False)

print(f"Excel file saved ✅")
print(f"Location: {output_path}")

Excel file saved ✅
Location: C:\Users\Sonu\Documents\hospital_report.xlsx


In [ ]:
wb = load_workbook(output_path)

# Define styles
header_font  = Font(bold=True, color="FFFFFF", size=11)
header_fill  = PatternFill("solid", fgColor="1F4E79")
alt_fill     = PatternFill("solid", fgColor="D6E4F0")
center_align = Alignment(horizontal="center", vertical="center")
border_side  = Side(style="thin", color="BBBBBB")
cell_border  = Border(
    left=border_side, right=border_side,
    top=border_side,  bottom=border_side
)

def format_sheet(ws):
    # Style header row
    for cell in ws[1]:
        cell.font      = header_font
        cell.fill      = header_fill
        cell.alignment = center_align
        cell.border    = cell_border

    # Style data rows with alternating colors
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        fill = alt_fill if row_idx % 2 == 0 else PatternFill()
        for cell in row:
            cell.fill      = fill
            cell.border    = cell_border
            cell.alignment = Alignment(vertical="center")

    # Auto fit column widths
    for col in ws.columns:
        max_len    = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            if cell.value:
                max_len = max(max_len, len(str(cell.value)))
        ws.column_dimensions[col_letter].width = min(max_len + 4, 40)

    # Freeze header row
    ws.freeze_panes = "A2"


# Apply to all sheets
for sheet_name in wb.sheetnames:
    format_sheet(wb[sheet_name])
    print(f"Formatted: {sheet_name} ✅")

wb.save(output_path)
print(f"\nFormatted file saved ✅")
print(f"Check your Desktop for hospital_report.xlsx")

Formatted: Monthly Trend ✅
Formatted: Revenue All Months ✅
Formatted: Doctor Performance ✅
Formatted: Unpaid Bills ✅
Formatted: Appointment Summary ✅
Formatted: Patient Age Groups ✅

Formatted file saved ✅
Check your Desktop for hospital_report.xlsx


In [ ]:
wb.save(output_path)
print(f"\nFormatted file saved ✅")
print(f"Check your Documents for hospital_report.xlsx")


Formatted file saved ✅
Check your Documents for hospital_report.xlsx


In [ ]:
connection.close()
print("Connection closed ✅")

Connection closed ✅
